# Importing libraries

In [19]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

from datasets import load_dataset

from sklearn.metrics import precision_score, recall_score, f1_score, hamming_loss
from sklearn.preprocessing import MultiLabelBinarizer
from memory_profiler import memory_usage
from time import perf_counter

# Importing dataset

In [20]:
ds = load_dataset("TimSchopf/arxiv_categories", "default")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,id,title,abstract,categories,creation_date
0,2204.14117,A Comparative Study of Meter Detection Methods...,In order to read meter values from a camera on...,[Computer Science Archive->cs.CV],2022-04-24 13:59:57+00:00
1,2305.19887,The Markov chain embedding problem in a low ju...,We consider the problem of finding the transit...,[Mathematics Archive->math.PR],2023-05-31 14:24:25+00:00
2,0910.5857,Chaotic Transport and Chronology of Complex As...,We present a transport model that describes th...,[Physics Archive->astro-ph->astro-ph.EP],2009-10-30 12:34:26+00:00
3,1801.10207,FITing-Tree: A Data-aware Index Structure,Index structures are one of the most important...,[Computer Science Archive->cs.DB],2018-01-30 20:22:53+00:00
4,0803.0849,The Universal Cardinal Ordering of Fixed Points,"We present the theorem which determines, by a ...",[Physics Archive->nlin->nlin.CD],2008-03-06 12:55:48+00:00
...,...,...,...,...,...
163163,1805.11049,Induced Chern-Simons modified gravity at finit...,We calculate the linearized four-dimensional g...,"[Physics Archive->gr-qc, Physics Archive->hep-...",2018-05-28 17:00:59+00:00
163164,1907.11966,Small Time Behavior and Summability for the Sc...,We consider the Carleson's problem regarding s...,[Mathematics Archive->math.AP],2019-07-27 18:59:04+00:00
163165,1510.08071,GM2Calc: Precise MSSM prediction for $(g - 2)$...,"We present GM2Calc, a public C++ program for t...",[Physics Archive->hep->hep-ph],2015-10-27 20:09:29+00:00
163166,1803.01475,"The Fu-Yau equation on compact astheno-K\""ahle...","In this paper, we study the Fu-Yau equation on...","[Mathematics Archive->math.AP, Mathematics Arc...",2018-03-05 02:54:16+00:00


# Dataset preprocessing

In [21]:
train_df = train_df.rename(columns={'title': 'text'})
val_df = val_df.rename(columns={'title': 'text'})
test_df = test_df.rename(columns={'title': 'text'})

train_df = train_df.rename(columns={'categories': 'labels'})
val_df = val_df.rename(columns={'categories': 'labels'})
test_df = test_df.rename(columns={'categories': 'labels'})

In [22]:
allowed_categories = ["cs.AI", "cs.CL", "stat.ML", "math.OC", "cs.LG"]

def clean_element(lst):
    final = []
    for elem in lst:
        clean = elem.split('->')[-1]
        final.append(clean)
    return final

train_df['labels'] = train_df['labels'].apply(clean_element)
val_df['labels'] = val_df['labels'].apply(clean_element)
test_df['labels'] = test_df['labels'].apply(clean_element)

In [23]:
train_df = train_df[train_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
test_df = test_df[test_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
val_df = val_df[val_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]

train_df = train_df[train_df['labels'].apply(len) > 0]
test_df = test_df[test_df['labels'].apply(len) > 0]
val_df = val_df[val_df['labels'].apply(len) > 0]

In [24]:
train_df.drop(columns=['id','abstract','creation_date'], inplace=True)
test_df.drop(columns=['id','abstract','creation_date'], inplace=True)
val_df.drop(columns=['id','abstract','creation_date'], inplace=True)

train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

train_df

,text,labels
0,Upper and Lower Bounds for Large Scale Multist...,[math.OC]
1,Binary Classification: Counterbalancing Class ...,[cs.LG]
2,Smooth Optimization with Approximate Gradient,[math.OC]
3,An AI-powered Smart Routing Solution for Payme...,[cs.AI]
4,A linearly convergent method for solving high-...,[math.OC]
...,...,...
10141,Simple Question Answering with Subgraph Rankin...,"[cs.CL, cs.LG, stat.ML]"
10142,"Fire Now, Fire Later: Alarm-Based Systems for ...","[cs.AI, cs.LG, stat.ML]"
10143,NSP-BERT: A Prompt-based Few-Shot Learner Thro...,"[cs.AI, cs.CL]"
10144,Near-optimal bounds for phase synchronization,[math.OC]


In [25]:
mlb = MultiLabelBinarizer()

train_labels_binarized = mlb.fit_transform(train_df['labels'])
val_labels_binarized = mlb.transform(val_df['labels'])
test_labels_binarized = mlb.transform(test_df['labels'])

train_labels_df = pd.DataFrame(train_labels_binarized, columns=mlb.classes_)
val_labels_df = pd.DataFrame(val_labels_binarized, columns=mlb.classes_)
test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

train_df = pd.concat([train_df, train_labels_df], axis=1)
val_df = pd.concat([val_df, val_labels_df], axis=1)
test_df = pd.concat([test_df, test_labels_df], axis=1)

train_df = train_df.drop(columns=['labels'])
val_df = val_df.drop(columns=['labels'])
test_df = test_df.drop(columns=['labels'])

train_df

,text,cs.AI,cs.CL,cs.LG,math.OC,stat.ML
0,Upper and Lower Bounds for Large Scale Multist...,0,0,0,1,0
1,Binary Classification: Counterbalancing Class ...,0,0,1,0,0
2,Smooth Optimization with Approximate Gradient,0,0,0,1,0
3,An AI-powered Smart Routing Solution for Payme...,1,0,0,0,0
4,A linearly convergent method for solving high-...,0,0,0,1,0
...,...,...,...,...,...,...
10141,Simple Question Answering with Subgraph Rankin...,0,1,1,0,1
10142,"Fire Now, Fire Later: Alarm-Based Systems for ...",1,0,1,0,1
10143,NSP-BERT: A Prompt-based Few-Shot Learner Thro...,1,1,0,0,0
10144,Near-optimal bounds for phase synchronization,0,0,0,1,0


In [26]:
class TextDataset(Dataset):
    def __init__(self, texts, label_matrix, tokenizer, max_len):
        """
        texts: a pandas Series or list of strings
        label_matrix: a pandas DataFrame or 2D NumPy array of shape [num_samples, num_labels]
                      Each row i has the 0/1 labels for text i.
        tokenizer: a transformers tokenizer
        max_len: maximum sequence length
        """
        self.texts = texts.tolist()
        # Convert the label matrix into a NumPy array if it isn't already
        self.labels = label_matrix.values if hasattr(label_matrix, 'values') else label_matrix
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        labels = self.labels[idx]  # shape: [num_labels]

        # Tokenize
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            # Convert labels to float so it works with BCEWithLogitsLoss
            'labels': torch.tensor(labels, dtype=torch.float)
        }


In [27]:
MAX_LEN = 128
BATCH_SIZE = 32

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

label_cols = [col for col in train_df.columns if col != 'text']

# Create datasets
train_dataset = TextDataset(
    texts=train_df['text'],
    label_matrix=train_df[label_cols],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df['text'],
    label_matrix=val_df[label_cols],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df['text'],
    label_matrix=test_df[label_cols],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Neural network class (LSTM)

In [28]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(LSTMClassifier, self).__init__()

        # For multi-label, output_dim = number_of_labels
        self.embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout
        )

        # If bidirectional=True, final hidden state has 2*hidden_dim
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        outputs, (hidden, cell) = self.lstm(embedded)

        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        hidden = self.dropout(hidden)
        logits = self.fc(hidden)  # shape [batch_size, output_dim]

        return logits  # raw logits for each label


# Instancing the LSTM model, criterion and optimizer

In [29]:
embedding_dim = 128
hidden_dim = 128
output_dim = len(label_cols)
n_layers = 2
bidirectional = True
dropout = 0.3

model = LSTMClassifier(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)

In [30]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')
model = model.to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Using cuda device


# Training and evaluation functions

In [31]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    losses = []
    correct_predictions = 0
    total_labels = 0

    all_labels = []
    all_preds = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)  # shape: (batch_size, num_labels)

        optimizer.zero_grad()

        # Forward pass -> logits: [batch_size, num_labels]
        logits = model(input_ids)

        # Compute loss
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        # Convert logits to predictions in {0,1}
        preds = (torch.sigmoid(logits) > 0.5).float()

        # Count how many individual labels are predicted correctly
        correct_predictions += (preds == labels).sum().item()
        total_labels += labels.numel()

        # Store for metric calculation
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    # Calculate mean loss
    avg_loss = sum(losses) / len(losses)
    # Label-level accuracy
    accuracy = correct_predictions / total_labels

    # Convert to NumPy
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    # Macro-average precision, recall, F1
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    # Per-class F1
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

    # Hamming loss
    ham_loss = hamming_loss(all_labels, all_preds)

    return accuracy, avg_loss, precision, recall, f1_macro, f1_per_class, ham_loss


def eval_model(model, data_loader, criterion, device):
    model.eval()
    losses = []
    correct_predictions = 0
    total_labels = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)
            losses.append(loss.item())

            preds = (torch.sigmoid(logits) > 0.5).float()

            correct_predictions += (preds == labels).sum().item()
            total_labels += labels.numel()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    avg_loss = sum(losses) / len(losses)
    accuracy = correct_predictions / total_labels

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)
    ham_loss = hamming_loss(all_labels, all_preds)

    return accuracy, avg_loss, precision, recall, f1_macro, f1_per_class, ham_loss


# Training loop

In [32]:
def training_loop(epochs):
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        
        (
            train_acc, 
            train_loss, 
            train_prec, 
            train_rec, 
            train_f1_macro, 
            train_f1_per_class,
            train_ham_loss
        ) = train_epoch(model, train_loader, optimizer, criterion, device)
        
        (
            val_acc, 
            val_loss, 
            val_prec, 
            val_rec, 
            val_f1_macro, 
            val_f1_per_class,
            val_ham_loss
        ) = eval_model(model, val_loader, criterion, device)
        
        print(f"Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, "
              f"Precision(macro): {train_prec:.4f}, Recall(macro): {train_rec:.4f}, "
              f"F1(macro): {train_f1_macro:.4f}, Hamming: {train_ham_loss:.4f}")
        print(f"F1 Per Class (Train): {train_f1_per_class}")
        
        print(f"Val   Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, "
              f"Precision(macro): {val_prec:.4f}, Recall(macro): {val_rec:.4f}, "
              f"F1(macro): {val_f1_macro:.4f}, Hamming: {val_ham_loss:.4f}")
        print(f"F1 Per Class (Val):   {val_f1_per_class}")
        print("--------------------------------------------------")
    
    return (
        train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class, train_ham_loss,
        val_acc,   val_loss,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class,   val_ham_loss
    )


In [33]:

seeds = [2, 3, 5]
EPOCHS = 5

# Update your results DataFrame with Hamming Loss columns
results = pd.DataFrame(columns=[
    'seed',
    'train_loss', 'train_acc', 'train_prec', 'train_rec', 'train_f1', 'train_f1_per_class', 'train_ham',
    'val_loss',   'val_acc',   'val_prec',   'val_rec',   'val_f1',   'val_f1_per_class',   'val_ham',
    'test_loss',  'test_acc',  'test_prec',  'test_rec',  'test_f1',  'test_f1_per_class',  'test_ham',
    'max_memory_usage_train', 'max_vram_usage_train', 'total_time_train',
    'max_memory_usage_test',  'max_vram_usage_test',  'total_time_test'
])

for seed in seeds:
    torch.manual_seed(seed)
    
    # Reset / re-initialize model for each seed
    model = LSTMClassifier(
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=len(label_cols),  # Number of labels for multi-label
        n_layers=n_layers,
        bidirectional=bidirectional,
        dropout=dropout
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    # For multi-label classification, use BCEWithLogitsLoss
    criterion = nn.BCEWithLogitsLoss().to(device)
    
    # Reset CUDA memory tracking if using GPU
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    # -------- TRAINING -----------
    start_time_train = perf_counter()
    
    max_memory_usage_train, retval = memory_usage(
        (training_loop, (EPOCHS,), {}),
        retval=True, 
        max_usage=True
    )
    total_time_train = perf_counter() - start_time_train

    max_vram_usage_train = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if torch.cuda.is_available() else None
    )

    (
        train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class, train_ham,
        val_acc,   val_loss,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class,   val_ham
    ) = retval

    # Reset CUDA memory tracking before test evaluation
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    # -------- TESTING -----------
    start_time_test = perf_counter()
    # eval_model should return:
    # (test_acc, test_loss, test_prec, test_rec, test_f1, test_f1_per_class, test_ham)
    max_memory_usage_test, retval = memory_usage(
        (eval_model, (model, test_loader, criterion, device), {}),
        retval=True, 
        max_usage=True
    )
    total_time_test = perf_counter() - start_time_test

    max_vram_usage_test = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if torch.cuda.is_available() else None
    )

    test_acc, test_loss, test_prec, test_rec, test_f1, test_f1_per_class, test_ham = retval

    # -------- LOGGING -----------
    new_row = pd.DataFrame([[
        seed,
        train_loss, train_acc, train_prec, train_rec, train_f1_macro, train_f1_per_class, train_ham,
        val_loss,   val_acc,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class,   val_ham,
        test_loss,  test_acc,  test_prec,  test_rec,  test_f1,        test_f1_per_class,  test_ham,
        max_memory_usage_train, max_vram_usage_train, total_time_train,
        max_memory_usage_test,  max_vram_usage_test,  total_time_test
    ]], columns=results.columns)

    results = pd.concat([results, new_row], ignore_index=True)

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.4905, Accuracy: 0.7639, Precision(macro): 0.5742, Recall(macro): 0.3380, F1(macro): 0.3962, Hamming: 0.2361
F1 Per Class (Train): [0.00512445 0.60582307 0.69459485 0.4833652  0.19205505]
Val   Loss: 0.4106, Accuracy: 0.8079, Precision(macro): 0.6884, Recall(macro): 0.4242, F1(macro): 0.4566, Hamming: 0.1921
F1 Per Class (Val):   [0.         0.80632411 0.76415094 0.68783069 0.0247678 ]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.3785, Accuracy: 0.8248, Precision(macro): 0.7103, Recall(macro): 0.5364, F1(macro): 0.5721, Hamming: 0.1752
F1 Per Class (Train): [0.01097695 0.85344283 0.79369458 0.75419171 0.44840352]
Val   Loss: 0.3951, Accuracy: 0.8201, Precision(macro): 0.8127, Recall(macro): 0.5213, F1(macro): 0.5646, Hamming: 0.1799
F1 Per Class (Val):   [0.02352941 0.81309686 0.78132296 0.71921182 0.48598131]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.3342, Accuracy: 0.8495, Precision(macro): 0.7681, 

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
C:\Users\Rafael\AppData\Local\Temp\ipykernel_8212\3822712177.py:87: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_row], ignore_index=True)
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.4905, Accuracy: 0.7623, Precision(macro): 0.5905, Recall(macro): 0.3438, F1(macro): 0.4009, Hamming: 0.2377
F1 Per Class (Train): [0.00731261 0.62805663 0.67365835 0.51837817 0.1769182 ]
Val   Loss: 0.4047, Accuracy: 0.8148, Precision(macro): 0.6275, Recall(macro): 0.4729, F1(macro): 0.5318, Hamming: 0.1852
F1 Per Class (Val):   [0.         0.8069241  0.76521739 0.6961326  0.39055794]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.3845, Accuracy: 0.8240, Precision(macro): 0.7177, Recall(macro): 0.5335, F1(macro): 0.5765, Hamming: 0.1760
F1 Per Class (Train): [0.04726631 0.84485801 0.78729908 0.75699466 0.44597811]
Val   Loss: 0.3842, Accuracy: 0.8223, Precision(macro): 0.7536, Recall(macro): 0.6075, F1(macro): 0.6063, Hamming: 0.1777
F1 Per Class (Val):   [0.05730659 0.84170854 0.79878049 0.72774869 0.60574413]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.3350, Accuracy: 0.8483, Precision(macro): 0.7633, 

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.4851, Accuracy: 0.7666, Precision(macro): 0.5751, Recall(macro): 0.3464, F1(macro): 0.4043, Hamming: 0.2334
F1 Per Class (Train): [0.00293794 0.61968761 0.68342298 0.54525547 0.16997721]
Val   Loss: 0.4185, Accuracy: 0.8060, Precision(macro): 0.6018, Recall(macro): 0.4538, F1(macro): 0.5045, Hamming: 0.1940
F1 Per Class (Val):   [0.         0.78461538 0.75220529 0.69918699 0.28638498]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.3837, Accuracy: 0.8225, Precision(macro): 0.7074, Recall(macro): 0.5282, F1(macro): 0.5650, Hamming: 0.1775
F1 Per Class (Train): [0.02030457 0.84438816 0.79352941 0.75336885 0.41342671]
Val   Loss: 0.3840, Accuracy: 0.8265, Precision(macro): 0.6044, Recall(macro): 0.5755, F1(macro): 0.5872, Hamming: 0.1735
F1 Per Class (Val):   [0.         0.85579196 0.80562061 0.72774869 0.54660348]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.3334, Accuracy: 0.8512, Precision(macro): 0.7725, 

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [36]:
results.to_csv('results/lstm_multilabel3.csv', index=False)
results.head()

,seed,train_loss,train_acc,train_prec,train_rec,train_f1,train_f1_per_class,train_ham,val_loss,val_acc,...,test_rec,test_f1,test_f1_per_class,test_ham,max_memory_usage_train,max_vram_usage_train,total_time_train,max_memory_usage_test,max_vram_usage_test,total_time_test
0,2,0.265477,0.886911,0.833927,0.742285,0.778533,"[0.5033396946564885, 0.9356743318831573, 0.870...",0.113089,0.393332,0.831754,...,0.658747,0.676097,"[0.36554621848739494, 0.8781725888324873, 0.79...",0.166772,2282.363281,231.645020,18.845771,2282.328125,194.172363,0.786214
1,3,0.259945,0.889966,0.839360,0.751758,0.787083,"[0.5461340808222378, 0.9357912826120919, 0.879...",0.110034,0.379860,0.836493,...,0.626172,0.662265,"[0.39923954372623577, 0.8634146341463415, 0.79...",0.170866,2282.113281,232.115723,18.545318,2281.335938,195.021484,0.735513
2,5,0.267760,0.884802,0.825540,0.746492,0.778559,"[0.5353022611905861, 0.9369649805447471, 0.871...",0.115198,0.385491,0.833965,...,0.632618,0.662332,"[0.3714821763602251, 0.8617283950617284, 0.798...",0.172913,2282.593750,231.314453,18.190016,2281.363281,194.357422,0.729500


In [37]:
torch.save(model.state_dict(), 'results/lstm_multilabel3.pth')